In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/12 00:03:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:

STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "mongo"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-09-06-Sep"
TABLE_NAME = "videos"

MONGO_BRONZE_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}/{TABLE_NAME}.parquet"

In [3]:
# ============================================================
# 1. LOAD SILVER DATASETS ONCE
# ============================================================

df_users_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/users.parquet")
df_subscriptions_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/subscription.parquet")
df_payments_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/payments.parquet")
df_videos_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/mongo/videos_silver.parquet")
df_kafka_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/kafka/events_silver.parquet")


26/10/12 00:04:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [18]:
df_videos_silver.select("video_id").count()

10000

In [19]:
df_kafka_silver.select("video_id").count()

15680

In [6]:
gold_dim_videos.select("video_id").count()

6342

In [7]:
gold_fact_events.select("video_id").count()

10471

In [4]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-gold"
TABLE_NAME_du = "dim_users"
TABLE_NAME_dv = "dim_videos"
TABLE_NAME_fs = "fact_subscription"
TABLE_NAME_fp = "fact_payments"
TABLE_NAME_fe = "fact_events"

In [5]:
dimu_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_du}"

In [6]:
dimv_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_dv}"
facts_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_fs}"
factp_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_fp}"
facte_path = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{TABLE_NAME_fe}"

In [7]:
gold_dim_users = spark.read.parquet(dimu_path)

In [8]:
gold_dim_videos = spark.read.parquet(dimv_path)

In [9]:
gold_fact_subscriptions = spark.read.parquet(facts_path)

In [10]:
gold_fact_payments = spark.read.parquet(factp_path)

In [11]:
gold_fact_events = spark.read.parquet(facte_path)

In [12]:
fk_tests = {
    "fact_subscriptions": ["user_id"],
    "fact_payments": ["subscription_id"],
    "fact_events": ["user_id", "video_id"],
}

for table_name, fks in fk_tests.items():
    df = globals()[f"gold_{table_name}"]

    for fk in fks:
        null_count = df.filter(F.col(fk).isNull()).count()

        print(
            f"{table_name}.{fk} -> "
            f"NULLs: {null_count}"
        )

fact_subscriptions.user_id -> NULLs: 0
fact_payments.subscription_id -> NULLs: 0
fact_events.user_id -> NULLs: 0
fact_events.video_id -> NULLs: 0


In [13]:
pk_tests = {
    "dim_users": "user_id",
    "dim_videos": "video_id",
    "fact_subscriptions": "subscription_id",
    "fact_payments": "payment_id",
    "fact_events": "event_id",
}

for table_name, pk in pk_tests.items():
    df = globals()[f"gold_{table_name}"]

    null_count = df.filter(F.col(pk).isNull()).count()

    print(
        f"{table_name}.{pk} -> "
        f"NULLs: {null_count}"
    )

dim_users.user_id -> NULLs: 0
dim_videos.video_id -> NULLs: 0
fact_subscriptions.subscription_id -> NULLs: 0
fact_payments.payment_id -> NULLs: 0
fact_events.event_id -> NULLs: 0


In [14]:
for table_name, pk in pk_tests.items():
    df = globals()[f"gold_{table_name}"]

    total_count = df.count()
    distinct_count = df.select(pk).distinct().count()

    print(
        f"{table_name}.{pk} -> "
        f"rows: {total_count}, "
        f"distinct: {distinct_count}, "
        f"unique: {total_count == distinct_count}"
    )

dim_users.user_id -> rows: 9997, distinct: 9997, unique: True
dim_videos.video_id -> rows: 6235, distinct: 6235, unique: True
fact_subscriptions.subscription_id -> rows: 20140, distinct: 20140, unique: True
fact_payments.payment_id -> rows: 11767, distinct: 11767, unique: True
fact_events.event_id -> rows: 8796, distinct: 8796, unique: True


In [15]:
fk_tests = {
    "fact_subscriptions": ["user_id"],
    "fact_payments": ["subscription_id"],
    "fact_events": ["user_id", "video_id"],
}

for table_name, fks in fk_tests.items():
    df = globals()[f"gold_{table_name}"]

    for fk in fks:
        null_count = df.filter(F.col(fk).isNull()).count()

        print(
            f"{table_name}.{fk} -> "
            f"NULLs: {null_count}"
        )

fact_subscriptions.user_id -> NULLs: 0
fact_payments.subscription_id -> NULLs: 0
fact_events.user_id -> NULLs: 0
fact_events.video_id -> NULLs: 0


In [16]:
relationships = [
    (
        gold_fact_subscriptions,
        "user_id",
        gold_dim_users,
        "user_id",
    ),
    (
        gold_fact_payments,
        "subscription_id",
        gold_fact_subscriptions,
        "subscription_id",
    ),
    (
        gold_fact_events,
        "user_id",
        gold_dim_users,
        "user_id",
    ),
    (
        gold_fact_events,
        "video_id",
        gold_dim_videos,
        "video_id",
    ),
]

for child_df, child_fk, parent_df, parent_pk in relationships:

    orphan_count = (
        child_df
        .join(
            parent_df.select(parent_pk),
            child_df[child_fk] == parent_df[parent_pk],
            "left_anti",
        )
        .count()
    )

    print(
        f"{child_fk} -> {parent_pk} | "
        f"orphans: {orphan_count}"
    )

user_id -> user_id | orphans: 6
subscription_id -> subscription_id | orphans: 1
user_id -> user_id | orphans: 5
video_id -> video_id | orphans: 4763


In [28]:
expected_columns = {
    "dim_users": [
        "user_id",
        "email",
        "country",
        "account_status",
        "created",
        "last_login",
    ],

    "dim_videos": [
        "video_id",
        "content_type",
        "match_id",
        "competition_name",
        "title",
        "languages",
        "duration_sec",
        "max_resolution",
        "status",
        "views",
        "likes",
        "created_at",
        "last_updated",
    ],

    "fact_subscriptions": [
        "subscription_id",
        "user_id",
        "plan_type",
        "start_date",
        "end_date",
        "is_active",
        "auto_renew",
    ],

    "fact_payments": [
        "payment_id",
        "subscription_id",
        "amount",
        "currency",
        "payment_timestamp",
        "payment_status",
    ],

    "fact_events": [
        "event_id",
        "event_timestamp",
        "user_id",
        "video_id",
        "interaction_type",
        "watch_time_sec",
        "device_type",
    ],
}

In [29]:
for table_name, expected in expected_columns.items():

    df = globals()[f"gold_{table_name}"]

    actual = df.columns

    missing = [c for c in expected if c not in actual]
    unexpected = [c for c in actual if c not in expected]

    print(f"\n{table_name}")

    print(f"Missing: {missing}")
    print(f"Unexpected: {unexpected}")
    print(f"Schema OK: {not missing and not unexpected}")


dim_users
Missing: ['created']
Unexpected: ['created_at']
Schema OK: False

dim_videos
Missing: ['languages', 'max_resolution', 'views', 'likes']
Unexpected: ['technical_max_resolution', 'stats_views', 'stats_likes', 'language_1', 'language_2']
Schema OK: False

fact_subscriptions
Missing: []
Unexpected: []
Schema OK: True

fact_payments
Missing: ['payment_timestamp']
Unexpected: ['payment_date']
Schema OK: False

fact_events
Missing: []
Unexpected: []
Schema OK: True


In [8]:
gold_fact_events \
    .join(
        gold_dim_videos.select("video_id"),
        gold_fact_events["video_id"] == gold_dim_videos["video_id"],
        "left_anti"
    ) \
    .select("video_id") \
    .groupBy("video_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

+------------------------------------+-----+
|video_id                            |count|
+------------------------------------+-----+
|01a076ee-55a2-74ce-a16b-333d2c21cc02|6    |
|01a076ee-4b40-7091-92db-438548eda26e|5    |
|01a076ee-4701-7a34-aad6-520989028b82|5    |
|01a076ee-6448-7f76-bca9-35bd4a934b0d|5    |
|01a076ee-5df2-75d4-8134-9a02e5cf41a7|5    |
|01a076ee-5013-7a92-9e39-7b5a8def5080|5    |
|01a076ee-5dcf-7b1a-a2fe-1555be1a5860|5    |
|01a076ee-5530-7874-88fd-e11bf4c1f84e|5    |
|01a076ee-5183-7bbf-b141-46a2278b6cb3|5    |
|01a076ee-5e92-7f6b-99ae-90051cf56fd4|4    |
|01a076ee-5982-7291-95dc-00504c7d5cc6|4    |
|01a076ee-5d03-7e08-9d27-ab19ed11c236|4    |
|01a076ee-5753-779b-a017-d3a31c5eab4a|4    |
|01a076ee-637b-76f6-af21-d5ab9f28b0c9|4    |
|01a076ee-646e-76f5-936d-dcdaae62628a|4    |
|01a076ee-479c-7342-aed3-e4f6b122aeb9|4    |
|01a076ee-6b76-796d-88ef-e62902467ac1|4    |
|01a076ee-4a2b-7597-9143-233019a848e5|4    |
|01a076ee-6b61-704e-a7b3-e473ea29435d|4    |
|01a076ee-

In [10]:
gold_fact_events \
    .join(
        gold_dim_videos.select("video_id"),
        gold_fact_events["video_id"] == gold_dim_videos["video_id"],
        "left_anti"
    ) \
    .count()

3020